# SUMAUTO EDA
 - Author   : Manuel Morello Martínez
 - Date     : 2026

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

## DATA ACQUISITION

In [8]:
PATH = r"C:/Users/manue/OneDrive/Escritorio/Zrive_DS/Final Project - KOMOREBI/KOMOREBI-PROJECT/data/"

df_advertiser       = pd.read_parquet(PATH + 'zrive_dim_advertiser.parquet')
df_monthly_snapshot = pd.read_parquet(PATH + 'zrive_fct_monthly_snapshot_advertiser.parquet')
df_withdrawal       = pd.read_parquet(PATH + 'zrive_advertiser_withdrawals.parquet')

In [10]:
#Nos quedamos sólo con aquellos anunciantes que tienen una antiguedad máxima de 3 meses (90 días)
date_cols = ["min_start_contrato_date", "updated_at"]
df_advertiser[date_cols] = df_advertiser[date_cols].apply(pd.to_datetime, errors="coerce")

df_advertiser["antiguedad_dias"] = (df_advertiser["updated_at"] - df_advertiser["min_start_contrato_date"]).dt.days

df_advertiser_lt_3m = df_advertiser[
    df_advertiser["antiguedad_dias"].notna()
    & (df_advertiser["antiguedad_dias"] >= 0)
    & (df_advertiser["antiguedad_dias"] < 90)
].copy()

print("Total advertisers:", df_advertiser["advertiser_zrive_id"].nunique())
print("Advertisers < 3 meses:", df_advertiser_lt_3m["advertiser_zrive_id"].nunique())

df_advertiser_lt_3m[["advertiser_zrive_id", "min_start_contrato_date", "updated_at", "antiguedad_dias"]].head(10)

Total advertisers: 7076
Advertisers < 3 meses: 796


,advertiser_zrive_id,min_start_contrato_date,updated_at,antiguedad_dias
1,6811,2025-01-24,2025-02-05 01:02:08,12
7,4642,2022-12-19,2023-03-07 01:13:14,78
9,3036,2024-02-02,2024-04-25 14:37:44,83
17,534,2024-04-30,2024-07-05 11:33:04,66
21,1291,2024-04-29,2024-07-05 11:30:50,67
36,6826,2025-02-04,2025-03-12 01:02:09,36
38,6054,2024-05-03,2024-07-05 11:35:32,63
43,833,2023-05-26,2023-06-26 01:00:11,31
44,5465,2023-09-27,2023-12-05 01:10:03,69
46,3442,2023-03-07,2023-05-09 12:31:11,63


In [ ]:
#Filtramos los dataframes para quedarnos sólo con los anunciantes con antiguedad < 3 meses
valid_ids = df_advertiser_lt_3m["advertiser_zrive_id"].dropna().unique()

df_monthly_snapshot_lt_3m = df_monthly_snapshot[
    df_monthly_snapshot["advertiser_zrive_id"].isin(valid_ids)
].copy()

df_withdrawal_lt_3m = df_withdrawal[
    df_withdrawal["advertiser_zrive_id"].isin(valid_ids)
].copy()

print(df_monthly_snapshot.shape[0], df_monthly_snapshot_lt_3m.shape[0])
print(df_monthly_snapshot["advertiser_zrive_id"].nunique(), df_monthly_snapshot_lt_3m["advertiser_zrive_id"].nunique())

print(df_withdrawal.shape[0], df_withdrawal_lt_3m.shape[0])
print(df_withdrawal["advertiser_zrive_id"].nunique(), df_withdrawal_lt_3m["advertiser_zrive_id"].nunique())

96829 4815
6968 740
22668 1119
6021 459


In [12]:
#Estudiamos como se comparten las variables mas importantes en los anunciantes con antiguedad < 3 meses
cols = df_monthly_snapshot_lt_3m.loc[:, "monthly_contracted_ads":"monthly_avg_ad_price"].columns

for c in cols:
    m = df_monthly_snapshot_lt_3m[c].mean()
    s = df_monthly_snapshot_lt_3m[c].std()
    print(f"{c}: mean={m:.4f} | std={s:.4f}")

monthly_contracted_ads: mean=119.6694 | std=346.5312
monthly_published_ads: mean=106.4820 | std=338.3848
monthly_unique_published_ads: mean=34.9688 | std=153.0278
monthly_distinct_ads: mean=181.5944 | std=549.4469
monthly_oro_ads: mean=1.8937 | std=15.8547
monthly_plata_ads: mean=1.1892 | std=2.3276
monthly_destacados_ads: mean=1.7875 | std=3.8872
monthly_pepitas_ads: mean=0.1917 | std=2.2011
monthly_shows: mean=85759.1739 | std=519879.7584
monthly_visits: mean=8582.9563 | std=45557.2277
monthly_leads: mean=21.0091 | std=54.5513
monthly_total_phone_views: mean=12.7215 | std=37.5532
monthly_total_calls: mean=9.3892 | std=26.4221
monthly_total_emails: mean=4.1867 | std=9.9475
monthly_total_invoice: mean=366.3145 | std=800.9576
monthly_total_reference_price: mean=1187.7210 | std=1783.4755
monthly_unique_calls: mean=6.4505 | std=16.7579
monthly_unique_emails: mean=9.6395 | std=23.3647
monthly_unique_leads: mean=16.0899 | std=39.0039
monthly_avg_ad_price: mean=33962.3646 | std=72681.0292
